# 🏀 NBA Foul Detection & Prediction — Complete End-to-End Interactive Pipeline
**Course:** CSC 434 — Artificial Intelligence & Machine Learning Final Project  
**Architecture:** Cross-Platform Dual-Stream Spatiotemporal `R(2+1)D-18` 3D CNN + `YOLOv11-Pose` 12-D Biomechanical Contact Encoder  

---
### Overview & How to Run Cell-by-Cell
This self-contained Jupyter Notebook consolidates the entire **4-stage NBA Foul Classifier pipeline** (`1_scrape_l2m.py`, `2_preprocess_tubes.py`, `3_model_and_dataset.py`, and `4_train_eval.py`) into one interactive, step-by-step workflow designed to run smoothly on **Windows, macOS (Intel & Apple Silicon M1/M2/M3/M4), and Linux (CUDA / CPU)**.

| Stage | Module Equivalent | Core Responsibility | Lightweight Default |
| :--- | :--- | :--- | :--- |
| **Cell 1** | Environment & Config | Cross-OS path resolution, device auto-selection (`CUDA` / `MPS` / `CPU`), & batch size knobs | `BATCH_SIZE = 2`, `ACCUM_STEPS = 2` |
| **Cell 2** | `1_scrape_l2m.py` | NBA Last Two Minute (L2M) scraper, UTF-8 BOM-safe parser, & ground-truth label mapper | `MAX_GAMES = 3`, `MAX_CLIPS = 12` |
| **Cell 3** | Exploratory Data Analysis | Inspects `l2m_metadata.csv` (200 plays, 13 games) & `l2m_incorrect_fouls.csv` (43 foul reviews) | Instant CSV summary |
| **Cell 4** | `2_preprocess_tubes.py` (Core) | `YOLOv11-Pose` 17-keypoint skeleton analysis, 12-D biomechanical contact features, & EMA ROI smoothing | `MAX_SCAN_SAMPLES = 12`, `YOLO_BATCH = 4` |
| **Cell 5** | `2_preprocess_tubes.py` (Runner) | Two-pass memory-safe action-tube extractor + offline synthetic `.pt` fallback generator | `MAX_PREPROCESS_CLIPS = 12` |
| **Cell 6** | `3_model_and_dataset.py` | `L2MFoulDataset`, `ConsistentVideoAugmentor`, `BiomechanicalFoulEncoder`, & `FoulClassifier3D` (`R(2+1)D-18`) | 512-D 3D CNN + 64-D Pose Fusion |
| **Cell 7** | `4_train_eval.py` (Train/Eval) | `FoulFocalBCEWithLogitsLoss` (`gamma=1.5`), gradient accumulation, & dynamic F1 threshold calibration | `EPOCHS = 5`, `MAX_TRAIN_SAMPLES = 20` |
| **Cell 8** | `4_train_eval.py` (Inference) | Single-clip inference, biomechanical contact telemetry plot, & annotated `.mp4` overlay export | Cross-OS codec fallback |

In [ ]:
# ==============================================================================
# CELL 1: Cross-Platform Environment Setup, Device Selection & Lightweight Config
# ==============================================================================
import os
import sys
import csv
import gc
import html
import json
import time
import random
import hashlib
import logging
import platform
import contextlib
from pathlib import Path
from collections import Counter
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

# Prevent OpenMP / MKL / OpenCV FFmpeg thread deadlocks on Windows, macOS, and Linux
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms.functional as TF
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

cv2.setNumThreads(0)

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

# Resolve project root directory automatically whether opened from repo root or NBA-Foul-Classifier-WIP/
CWD = Path.cwd().resolve()
if (CWD / "NBA-Foul-Classifier-WIP").exists():
    PROJECT_DIR = CWD / "NBA-Foul-Classifier-WIP"
else:
    PROJECT_DIR = CWD

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_CSV = DATA_DIR / "l2m_metadata.csv"
INCORRECT_CSV = DATA_DIR / "l2m_incorrect_fouls.csv"
PROCESSED_METADATA_CSV = PROCESSED_DIR / "processed_metadata.csv"
CHECKPOINT_PATH = PROJECT_DIR / "best_foul_r2plus1d.pt"

# ------------------------------------------------------------------------------
# LIGHTWEIGHT DEFAULT CONFIGURATION (Safe for 8GB RAM Laptops & CPU/MPS/CUDA)
# Increase these values when running on a workstation or dedicated GPU!
# ------------------------------------------------------------------------------
CONFIG = {
    "max_games": 3,              # Default: 3 games (Scale up to 25-100+ for full scraping)
    "max_scrape_clips": 50,      # Default: 12 MP4 clips downloaded (Scale up to 200-1000+)
    "max_preprocess_clips": 50,  # Default: 12 clips preprocessed (Set to 0 to process all 200+)
    "max_live_play_sec": 30.0,   # Scan first 30s of broadcast clip (skips multi-minute replay reviews)
    "max_scan_samples": 12,      # Default: 12 pose scan frames per clip (Scale up to 24-32)
    "yolo_batch_size": 4,        # Default: 4 frames per YOLO pose call (Scale up to 8-16 on GPU)
    "num_frames": 16,            # 16 frames sampled over 1.5s window around peak foul contact
    "crop_size": 224,            # 224x224 spatial action-tube crop
    "batch_size": 3,             # Micro-batch size per forward pass (Scale up to 4-8 on GPU)
    "accum_steps": 2,            # Gradient accumulation steps (Effective batch size = 2 * 2 = 4)
    "epochs": 10,                 # Default: 5 epochs for fast runs (Scale up to 20-30 for convergence)
    "max_train_samples": 20,     # Default: 20 samples (Set to 0 to train on full dataset)
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "dropout": 0.4,
    "seed": 42,
}

def select_compute_device(requested: Optional[str] = None) -> torch.device:
    if requested and requested.lower() != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def get_safe_num_workers(requested_workers: int = 0) -> int:
    # Windows and macOS use 'spawn' for multiprocessing; num_workers=0 avoids notebook/MPS hangs
    if platform.system() in ("Darwin", "Windows"):
        return 0
    return max(0, requested_workers)

DEVICE = select_compute_device()
print(f"Operating System : {platform.system()} ({platform.machine()})")
print(f"PyTorch Version  : {torch.__version__}")
print(f"Selected Device  : {DEVICE}")
print(f"Project Directory: {PROJECT_DIR}")

---
## Stage 1: NBA Last Two Minute (L2M) Scraper & Ground-Truth Labeler (`1_scrape_l2m.py`)
Official NBA Last Two Minute (L2M) reports evaluate every officiating decision inside the final 2 minutes of Q4 and Overtime:
- **Positive Class (`1 = foul`)**: `CC` (*Correct Call*) and `INC` (*Incorrect Non-Call* — an uncalled foul that actually occurred).
- **Negative Class (`0 = no_foul`)**: `CNC` (*Correct Non-Call* — legal guarding contact) and `IC` (*Incorrect Call* — a phantom whistle on legal defense).

Run the cell below to synchronize `data/l2m_incorrect_fouls.csv` from the cached `data/l2m_metadata.csv` (or set `RUN_LIVE_WEB_SCRAPE = True` to scrape a small batch of `MAX_GAMES = 3` games online).

In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — NBA L2M Scraper & Local Metadata Sync
# ==============================================================================
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
scrape_mod = importlib.import_module("1_scrape_l2m")

RUN_LIVE_WEB_SCRAPE = False  # Set True to query official.nba.com for up to CONFIG['max_games'] games

if RUN_LIVE_WEB_SCRAPE:
    scraped_rows = scrape_mod.run_training_dataset_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        max_games=CONFIG["max_games"],
        max_clips=CONFIG["max_scrape_clips"],
        skip_download=False,
    )
else:
    # Sync & populate l2m_incorrect_fouls.csv offline from cached l2m_metadata.csv
    populated_foul_rows = scrape_mod.populate_foul_csv_from_local_metadata(
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        include_confirmed_fouls=True,
    )
    print(f"Synced {len(populated_foul_rows)} foul review entries into {INCORRECT_CSV.name}")

---
## Exploratory Data Analysis (EDA): Inspecting the L2M Foul Dataset
Before running computer vision preprocessing, let's inspect the class balance, review decisions (`CNC`, `CC`, `INC`, `IC`), and foul types (`Shooting Foul` vs. `Personal Foul`) in `data/l2m_metadata.csv`.

In [ ]:
# ==============================================================================
# CELL 3: Dataset Summary & Class Imbalance Analysis
# ==============================================================================
with open(METADATA_CSV, "r", encoding="utf-8") as f:
    meta_rows = list(csv.DictReader(f))

label_counts = Counter(int(r["label"]) for r in meta_rows)
decision_counts = Counter(r["review_decision"] for r in meta_rows)
call_counts = Counter(r["call_type"] for r in meta_rows)
unique_games = sorted(set(r["game_id"] for r in meta_rows))

print(f"Total Labeled Plays in {METADATA_CSV.name}: {len(meta_rows)} across {len(unique_games)} NBA games")
print(f"Class Distribution : no_foul (0) = {label_counts[0]} ({label_counts[0]/len(meta_rows)*100:.1f}%) | "
      f"foul (1) = {label_counts[1]} ({label_counts[1]/len(meta_rows)*100:.1f}%)")
print(f"Imbalance Ratio    : {label_counts[0] / max(1, label_counts[1]):.2f} : 1 (no_foul : foul)")
print(f"Review Decisions   : {dict(decision_counts)}")
print(f"Foul Call Types    : {dict(call_counts)}")
print("\nSample plays from l2m_metadata.csv:")
for sample_row in meta_rows[:5]:
    print(f"  • [{sample_row['play_id']}] {sample_row['call_type']} ({sample_row['review_decision']} -> label={sample_row['label']}): "
          f"{sample_row['player_committing']} on {sample_row['player_disadvantaged']}")

---
## Stage 2: Biomechanical Pose Estimation & Memory-Safe Action-Tube Extraction (`2_preprocess_tubes.py`)
Why can't we feed full 1080p broadcast clips directly into a 3D CNN?
1. **RAM/VRAM Explosion**: Official L2M clips sometimes include multi-minute Secaucus Replay Center reviews (up to `12,318` frames / `238 MB`, which would allocate **76+ GB** of uncompressed 1080p frames in RAM).
2. **Spatial Dilution**: In a wide broadcast camera angle, the two interacting players occupy less than 8% of the pixels.

### Our Two-Pass Cross-Platform Solution:
- **Pass 1 (Lightweight Pose Scan)**: Caps scanning to the first `15.0` seconds of live play, decodes only `12` downscaled `640x360` frames (< 8 MB RAM), closes `cv2.VideoCapture` cleanly, and runs `YOLOv11-Pose` (`yolo11n-pose.pt`) in micro-batches (`yolo_batch_size = 4`).
- **12-D Biomechanical Contact Vector**: From the 17-keypoint human skeletons, we compute:
  - `[0]` Normalized wrist/elbow-to-arm proximity
  - `[1]` Defender arm-to-shooter upper-body (head/shoulders/wrists) intrusion
  - `[2]` Torso/hip collision proximity
  - `[3]` Player bounding-box IoU
  - `[4]` Vertical elevation velocity
  - `[5]` Defender lateral closing speed
  - `[6]` Shooter arm elevation above shoulders (shooting gather/release)
  - `[7]` Defender vertical contest elevation
  - `[8]` Basketball proximity to the interacting pair
  - `[9]` Local paint crowding density
  - `[10]` Vertical center offset
  - `[11]` Composite biomechanical foul-contact intensity score
- **Pass 2 (Stabilized 16-Frame Crop)**: Applies bidirectional Exponential Moving Average (EMA) smoothing to the ROI trajectory and decodes **only the 16 frames** in a `1.5s` window around peak contact at `224x224` resolution.

In [ ]:
# ==============================================================================
# CELL 4: Inspect Stage 2 Biomechanical Pose & Action-Tube Functions
# ==============================================================================
preprocess_mod = importlib.import_module("2_preprocess_tubes")
importlib.reload(preprocess_mod)

FEATURE_NAMES_12D = [
    "0: Arm-to-Arm Proximity",
    "1: Defender Arm -> Shooter Upper-Body Intrusion",
    "2: Torso/Hip Collision Proximity",
    "3: Pair Bounding-Box IoU",
    "4: Vertical Elevation Velocity",
    "5: Defender Closing Velocity",
    "6: Shooter Gather Arm Elevation",
    "7: Defender Verticality Arm Elevation",
    "8: Ball-to-Pair Proximity",
    "9: Paint Crowding Density",
    "10: Vertical Height Offset",
    "11: Composite Foul-Contact Score",
]
print("12-D Biomechanical Foul Feature Schema:")
for fname in FEATURE_NAMES_12D:
    print(f"  • {fname}")

In [ ]:
# ==============================================================================
# CELL 5: Run Stage 2 Preprocessing in a Small Batch (with Offline Fallback)
# ==============================================================================
# Processes up to CONFIG['max_preprocess_clips'] (12) balanced clips.
# If raw MP4 files are present in data/raw/, uses YOLOv11-Pose; otherwise uses
# deterministic synthetic action tubes so downstream training cells work offline!
processed_rows = preprocess_mod.preprocess_dataset(
    metadata_csv=METADATA_CSV,
    processed_dir=PROCESSED_DIR,
    processed_metadata_csv=PROCESSED_METADATA_CSV,
    window_sec=1.5,
    num_frames=CONFIG["num_frames"],
    crop_size=CONFIG["crop_size"],
    device=str(DEVICE),
    max_live_play_sec=CONFIG["max_live_play_sec"],
    max_scan_samples=CONFIG["max_scan_samples"],
    max_clips=CONFIG["max_preprocess_clips"],
    yolo_batch_size=CONFIG["yolo_batch_size"],
    synthetic_fallback=True,
)
print(f"\nPreprocessed {len(processed_rows)} balanced clips -> {PROCESSED_METADATA_CSV}")

---
## Stage 3: Dual-Stream Spatiotemporal Architecture (`3_model_and_dataset.py`)
Our classifier fuses two complementary representations:
1. **Visual Spatiotemporal Stream (`R(2+1)D-18`)**: Decomposes 3D convolutions into 2D spatial convolutions followed by 1D temporal convolutions over the `(3, 16, 224, 224)` action tube, producing a `512-D` embedding. Early layers (`stem`, `layer1`, `layer2`) are frozen to preserve general motion priors and reduce memory usage, while `layer3` and `layer4` are fine-tuned.
2. **Biomechanical Pose Stream (`BiomechanicalFoulEncoder`)**: A two-layer `Conv1d` network (`12 -> 32 -> 64` channels) with **Temporal Contact-Frame Attention Pooling** over the `(16, 12)` pose feature sequence, producing a `64-D` kinematic embedding focused on the exact moment of peak contact.
3. **Late Fusion Classification Head**: Concatenates `[512-D visual || 64-D pose] -> 576-D` through `LayerNorm -> Dropout(0.4) -> Linear(576, 128) -> GELU -> Dropout(0.2) -> Linear(128, 1)`.

In [ ]:
# ==============================================================================
# CELL 6: Verify Dataset Loading & Dual-Stream FoulClassifier3D Forward Pass
# ==============================================================================
model_mod = importlib.import_module("3_model_and_dataset")
importlib.reload(model_mod)

ConsistentVideoAugmentor = model_mod.ConsistentVideoAugmentor
L2MFoulDataset = model_mod.L2MFoulDataset
FoulClassifier3D = model_mod.FoulClassifier3D

demo_dataset = L2MFoulDataset(
    processed_dir=PROCESSED_DIR,
    metadata_csv=PROCESSED_METADATA_CSV,
    is_train=True,
    max_samples=CONFIG["max_train_samples"],
    return_foul_features=True,
)

sample_video, sample_feats, sample_label = demo_dataset[0]
print(f"Loaded Dataset Size  : {len(demo_dataset)} samples")
print(f"Sample Video Tensor  : shape={tuple(sample_video.shape)}, dtype={sample_video.dtype}")
print(f"Sample Pose Features : shape={tuple(sample_feats.shape)}, dtype={sample_feats.dtype}")
print(f"Sample Ground Truth  : label={int(sample_label.item())} ({'foul' if sample_label.item() == 1 else 'no_foul'})")

# Instantiate model (uses pretrained=False here for instant offline verification; set pretrained=True for full training)
test_model = FoulClassifier3D(pretrained=False, dropout_p=CONFIG["dropout"], freeze_early=True).to(DEVICE)
param_summary = test_model.trainable_parameter_summary()

with torch.no_grad():
    sample_logit = test_model(
        sample_video.unsqueeze(0).to(DEVICE),
        sample_feats.unsqueeze(0).to(DEVICE),
    )
    sample_prob = torch.sigmoid(sample_logit).item()

print(f"\nModel Parameter Breakdown:")
print(f"  • Total Parameters    : {param_summary['total_params']:,}")
print(f"  • Trainable Parameters: {param_summary['trainable_params']:,} (layer3, layer4, pose_encoder, classifier)")
print(f"  • Frozen Parameters   : {param_summary['frozen_params']:,} (stem, layer1, layer2)")
print(f"Forward Pass Check      : logit={sample_logit.item():.4f} -> P(foul)={sample_prob*100:.2f}%")

---
## Stage 4: Lightweight Training Loop, Focal Loss & Threshold Calibration (`4_train_eval.py`)
Because legal defensive contests (`CNC`) outnumber actual fouls (`CC`/`INC`) by roughly **3.65 : 1** in NBA L2M reports, standard unweighted Binary Cross-Entropy collapses toward predicting `no_foul`. We solve this using:
1. **Class-Balanced `FoulFocalBCEWithLogitsLoss` (`gamma = 1.5`)**: Multiplies positive examples by `pos_weight = N_neg / N_pos` and scales gradients by $(1 - p_t)^{\gamma}$ so subtle arm/wrist fouls receive strong gradient updates.
2. **Micro-Batch Gradient Accumulation (`batch_size = 2`, `accum_steps = 2`)**: Keeps peak memory low enough for any laptop while maintaining stable effective batch updates.
3. **Validation F1 Threshold Calibration**: Sweeps probability thresholds $\tau \in [0.25, 0.75]$ after every epoch to maximize validation F1-score.

In [ ]:
# ==============================================================================
# CELL 7: Train & Calibrate the Dual-Stream Foul Classifier
# ==============================================================================
train_mod = importlib.import_module("4_train_eval")
importlib.reload(train_mod)

train_mod.seed_everything(CONFIG["seed"])

all_samples = demo_dataset.samples
labels = [lbl for _, lbl in all_samples]
stratify_labels = labels if len(set(labels)) > 1 and min(labels.count(0), labels.count(1)) >= 2 else None

train_samples, val_samples = train_test_split(
    all_samples,
    test_size=0.25,
    random_state=CONFIG["seed"],
    stratify=stratify_labels,
)

train_ds = L2MFoulDataset(samples=train_samples, is_train=True, return_foul_features=True)
val_ds = L2MFoulDataset(samples=val_samples, is_train=False, return_foul_features=True)

safe_workers = get_safe_num_workers(0)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=safe_workers)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=safe_workers)

# Set USE_PRETRAINED_KINETICS=True when internet is available to download Kinetics-400 weights (~125MB)
USE_PRETRAINED_KINETICS = False
model = FoulClassifier3D(
    pretrained=USE_PRETRAINED_KINETICS,
    dropout_p=CONFIG["dropout"],
    freeze_early=True,
).to(DEVICE)

pos_weight = train_mod.compute_pos_weight(train_samples, device=DEVICE)
criterion = train_mod.FoulFocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=1.5)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, CONFIG["epochs"]), eta_min=CONFIG["lr"] * 0.01)
scaler = torch.amp.GradScaler("cuda", enabled=True) if (DEVICE.type == "cuda" and torch.cuda.is_available()) else None

history = []
best_f1 = -1.0

for epoch in range(1, CONFIG["epochs"] + 1):
    t_loss = train_mod.train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        accum_steps=CONFIG["accum_steps"],
    )
    v_loss, metrics = train_mod.evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": t_loss, "val_loss": v_loss, **metrics})
    print(
        f"Epoch [{epoch:02d}/{CONFIG['epochs']:02d}] | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
        f"Acc: {metrics['accuracy']:.3f} | F1: {metrics['f1']:.3f} | ROC-AUC: {metrics['roc_auc']:.3f} | "
        f"Optimal Thresh: {metrics['threshold']:.2f}"
    )
    if metrics["f1"] >= best_f1:
        best_f1 = metrics["f1"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "metrics": metrics,
                "optimal_threshold": metrics["threshold"],
            },
            CHECKPOINT_PATH,
        )

print(f"\nSaved calibrated checkpoint -> {CHECKPOINT_PATH}")

---
## Stage 5: Single-Clip Foul Prediction & Biomechanical Contact Telemetry
The cell below loads the saved `best_foul_r2plus1d.pt` checkpoint, evaluates a validation play, displays its frame-by-frame biomechanical contact profile across all 16 frames, and shows how to run `predict_single_clip()` on any raw `.mp4` broadcast file.

In [ ]:
# ==============================================================================
# CELL 8: Inference & Frame-by-Frame Biomechanical Telemetry Inspection
# ==============================================================================
ckpt = model_mod.load_torch_file_safe(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
calibrated_threshold = float(ckpt.get("optimal_threshold", 0.50))
model.eval()

val_path, val_true_label = val_samples[0]
val_video, val_feats, _ = val_ds[0]

with torch.no_grad():
    logit = model(val_video.unsqueeze(0).to(DEVICE), val_feats.unsqueeze(0).to(DEVICE))
    foul_prob = float(torch.sigmoid(logit).item())

verdict = "FOUL DETECTED" if foul_prob >= calibrated_threshold else "LEGAL CONTACT / NO FOUL"
peak_frame_idx = int(torch.argmax(val_feats[:, 11]).item())
peak_contact_score = float(val_feats[peak_frame_idx, 11].item())

print(f"Evaluated Clip      : {val_path.name}")
print(f"Ground-Truth Label  : {val_true_label} ({'FOUL' if val_true_label == 1 else 'NO FOUL'})")
print(f"Predicted Probability: {foul_prob * 100:.2f}% (Calibrated Threshold: {calibrated_threshold * 100:.0f}%)")
print(f"Model Verdict       : {verdict}")
print(f"Peak Contact Frame  : Frame {peak_frame_idx + 1}/16 (Biomechanical Score = {peak_contact_score * 100:.1f}%)")

# Optional: If a raw MP4 clip exists in data/raw/, generate an annotated MP4 video overlay
existing_mp4s = list(RAW_DIR.rglob("*.mp4"))
if existing_mp4s and YOLO is not None:
    sample_mp4 = existing_mp4s[0]
    print(f"\nRunning full MP4 video overlay inference on {sample_mp4.name}...")
    train_mod.predict_single_clip(video_path=sample_mp4, checkpoint_path=CHECKPOINT_PATH)
else:
    print("\n[Tip] To generate an annotated MP4 overlay on a raw broadcast video, run:")
    print("      python 4_train_eval.py --predict data/raw/foul/<play_id>.mp4")